# GBM internal val — pretrained pan-cancer Hist2Pheno

CODEX GBM, fold 0 is `P174511_Initial`. `P179161_Recurrent` is fold 1 and is not shown.

Checkpoint: `data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`  
This is the UNI2 A2 best model. That file is the fold-1 checkpoint, so these notebooks show **fold 0** instead of fold 1. Deployment skipped the 4×3 panel PNGs, so GT versus prediction is drawn from `matched_pred.h5ad`.

```bash
bash code/Internal_val/demo.sh status
ONLY=GBM bash code/Internal_val/demo.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

DATASET = "GBM"
SAVE_FIGURES = False
fold = iv.fold0_table(DATASET)
viz = iv.viz_samples(DATASET)
cov = iv.coverage(DATASET)
print(f"{DATASET} fold {iv.FOLD}: {len(fold)} sections; drawing {len(viz)}")
display(cov)
met = iv.metrics_fold0(DATASET)
if not met.empty:
    cols = [c for c in ("sample", "n_matched", "macro_auc_L1_cells", "macro_auc_L12_cells", "macro_auc_L2_cells") if c in met]
    display(met[cols])
print("viz", viz)


In [ ]:
display(Markdown("## Existing 4×3 panels (GT / Hist2Pheno / ROC / StarDist-all)"))
missing = []
for sample in viz:
    panel = iv.pred_paths(DATASET, sample)["panel"]
    if panel.is_file():
        display(Markdown(f"### {sample}"))
        display(Image(filename=str(panel)))
    else:
        missing.append(sample)
        print("missing panel", sample)
if missing:
    print("UNI2 A2 deployment did not save 4x3 panel PNGs. GT versus prediction is drawn from matched_pred.h5ad.")


In [ ]:
display(Markdown("## GT vs pretrained L2 (matched nuclei)"))
for sample in viz:
    pred = iv.pred_paths(DATASET, sample)["pred"]
    if not pred.is_file():
        print("skip", sample, "(no matched_pred.h5ad)")
        continue
    fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)
    fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES)
    display(fig)
    plt.close(fig)
